In [ ]:
import os
import sys
from pathlib import Path

# Local clone: run the notebook from anywhere inside the repository.
# On the VM, set EU to your name to import YOUR clone (notebooks/README.md).
EU = None
if EU:
    REPO = Path.home() / "shared" / "civicwater" / "users" / EU / "repo"
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sentinel").is_dir())
sys.path.insert(0, str(REPO / "src"))  # imports THIS clone

for v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[v] = "1"  # 3 CPUs for 6 people

# Local or VM: the data directory comes from SENTINEL_DATA_DIR (docs/data.md, section 1).
DATA_DIR = Path(
    os.environ.get("SENTINEL_DATA_DIR", Path.home() / "shared" / "civicwater" / "data")
).expanduser()
OUT_DIR = DATA_DIR / "p4_detect" / "dev-baseline-2018"  # outside git

%load_ext autoreload
%autoreload 2

import sentinel

print("sentinel from:", Path(sentinel.__file__).parent)
print("data dir:", DATA_DIR)

# P4 - development calibration on 2018 (BE-012f)

**Goal.** Learn the structure of the real residuals, test the sigma estimator and exercise the
`(h, N)` sweep, using the out-of-fold residuals of a local hour-of-week baseline on 2018
(ADR-008 section E). The pipeline lives in `sentinel.detection.offline.devcal`; this notebook
only sets the parameters, calls it and shows the tables. `report.md` in the output directory has
the same tables.

**Warnings (ADR-008 sections E and H).**

> simulated benchmark (BattLeDIM L-Town); thresholds fixed on 2018; development calibration
> (`dev-baseline-2018`, is_mock), not a delivered result; no 2019 file opened; matching rule
> provisional (Q-UI-06); delta/k not pre-registered

- 2018 is not a clean normal year: a leak is active in almost every window, so sigma is
  conservative and the labelled windows are only used for a sensitivity analysis.
- Everything is written outside git, under `$SENTINEL_DATA_DIR/p4_detect/`; the notebook has no
  saved outputs.
- No number below is a result of the final detector. The final calibration waits for the ML
  owner's residuals (DEP-DET-05).

In [ ]:
from datetime import timedelta

from sentinel.detection.offline import devcal
from sentinel.detection.offline.layer1 import MatchRule

# Every value in this block is a development choice, not pre-registered (ADR-008 section G).
SLOT_MINUTES = 60  # development choice, not pre-registered: baseline slot = day of week x hour
STATISTIC = "median"  # development choice, not pre-registered: baseline reduction ("median" or "mean")
SIGMA_FLOOR_M = 0.01  # development choice, not pre-registered: floor of sigma, metres
K = 0.5  # development choice, not pre-registered: delta = 2k = 1 sigma (textbook value, provisional)
H_VALUES = (4.0, 6.0, 8.0, 10.0, 12.0, 16.0)  # development choice, not pre-registered
N_VALUES = (1, 2, 3)  # development choice, not pre-registered
C = 2.0  # development choice, not pre-registered: S is capped at c*h
H_CLOSE_RATIO = 0.5  # development choice, not pre-registered: h_close = ratio * h
ARTEFACT_H = 8.0  # development choice, not pre-registered: h written to the artefact (in H_VALUES)
ARTEFACT_N = 2  # development choice, not pre-registered: N written to the artefact (in N_VALUES)
MATCH = MatchRule(  # development choice, not pre-registered; provisional rule (Q-UI-06)
    {"abrupt": timedelta(hours=48), "incipient": timedelta(days=30)}
)

In [ ]:
report = devcal.run(
    DATA_DIR,
    OUT_DIR,
    slot_minutes=SLOT_MINUTES,
    statistic=STATISTIC,
    sigma_floor_m=SIGMA_FLOOR_M,
    k=K,
    h_values=H_VALUES,
    n_values=N_VALUES,
    c=C,
    h_close_ratio=H_CLOSE_RATIO,
    artefact_h=ARTEFACT_H,
    artefact_n=ARTEFACT_N,
    match=MATCH,
)
print(f"done in {report.elapsed_seconds:.1f} s; files in {report.out_dir}")

In [ ]:
print("artefact:", report.artefact_path)
print("artefact sha256:", report.artefact_sha256)
print("residuals sha256 (uncompressed CSV):", report.residuals_sha256)
print("folds sha256:", report.folds_sha256)
print("provenance:", report.artefact.provenance, "| is_mock:", report.artefact.is_mock)

## Residual structure

Why the textbook i.i.d. CUSUM formulas do not apply (ADR-008 context): residuals are
autocorrelated, heteroscedastic through the day and correlated across sensors.
`z = (expected - observed) / sigma = -residual / sigma`.

In [ ]:
display(report.structure_ranges)  # min / median / max across the sensors
pair = report.pairwise
print(f"pairwise correlation ({pair.pairs} pairs): median {pair.median:.3f}, "
      f"max {pair.maximum:.3f} {pair.maximum_pair}")
print(f"mean correlation with the common mode: {report.structure['common_mode_corr'].mean():.3f}")
display(report.structure)  # one row per sensor

## Sensitivity of sigma to the leak labels

`todos` masks every window inside `[start, end]` of any labelled leak (the windows with an active
leak); it leaves about 2% of the year, so the ratio also reflects the period. `reparados`
(the P1 default) is the cross-check. `ratio = sigma_all / sigma_clean`.

In [ ]:
for policy, table in report.sigma_sensitivity.items():
    print(f"policy {policy}: {report.clean_fraction[policy]:.1%} of the windows are clean")
    display(table)

## Sweep over h and N

Counts n/N of the 14 labelled leaks of 2018 under the provisional matching rule; no confidence
intervals. The false alarms are not monotone in h: read the table, do not interpolate it.

In [ ]:
tables = devcal.summarise_sweep(report.sweep)
display(tables["counts"])
display(tables["delays_h"])  # median delay in hours
display(tables["missed"])